DEEP AGENTS

In [77]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["GROQ_API_KEY2"]=os.getenv("GROQ_API_KEY2")
os.environ["OPENAI_API_KEY2"]=os.getenv("OPENAI_API_KEY2")
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")

StateBackend

In [78]:
import os
from deepagents import create_deep_agent
from deepagents.backends import StateBackend


In [79]:
#create agent
agent=create_deep_agent(model="groq:qwen/qwen3.8-27b")
agent2=create_deep_agent(
    model="groq:qwen/qwen3.8-27b",
    backend=StateBackend()
)

In [80]:
result = agent2.invoke({
    "messages": [{
        "role": "user",
        "content": """create a file at /notes/todo.txt with exact same content:
1.read
2.write
3.upload
then tell u have done it"""
    }]
})

print("agent reply:")
print(result["messages"][-1].content)


agent reply:
I've created `/notes/todo.txt` with the exact content you specified:

```
1.read
2.write
3.upload
```

Done! ✅


In [81]:
result

{'messages': [HumanMessage(content='create a file at /notes/todo.txt with exact same content:\n1.read\n2.write\n3.upload\nthen tell u have done it', additional_kwargs={}, response_metadata={}, id='8a4f12b5-3bf5-4cc1-b20e-03239466e3f9'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '9vp53jkvq', 'function': {'arguments': '{"content":"1.read\\n2.write\\n3.upload","file_path":"/notes/todo.txt"}', 'name': 'write_file'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 2750, 'total_tokens': 2799, 'completion_time': 0.153670727, 'completion_tokens_details': None, 'prompt_time': 0.196422432, 'prompt_tokens_details': None, 'queue_time': 0.050200331, 'total_time': 0.350093159}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_a1293f40b5', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0e83f-5ff6-71a0-b7f2-e4203199b15e-0', tool_calls=[{'name': 

In [82]:
print("Backend check")
files = result.get("files", {})

if files:
    print(f"✅ StateBackend is working — {len(files)} file(s) in state:")
    for path, content in files.items():
        print(f"\n📄 {path}\n{'-' * 40}\n{content}")
else:
    print("⚠️  No files found in state. Either the agent didn't write a file, "
          "or the backend isn't wired up correctly.")

Backend check
✅ StateBackend is working — 1 file(s) in state:

📄 /notes/todo.txt
----------------------------------------
{'content': '1.read\n2.write\n3.upload', 'encoding': 'utf-8', 'created_at': '2026-09-28T13:41:03.245323+00:00', 'modified_at': '2026-09-28T13:41:03.245323+00:00'}


In [85]:
# 4. Prove persistence WITHIN the same thread:
# feed the returned state back in and ask i t to READ the file
followup = agent2.invoke({
    # carry forward prior messages + the files state
    "messages":  [{
        "role": "user",
        "content": "Read /notes/todo.txt."
    }],
    "files": result.get("files", {}),   # <-- pass the virtual filesystem along
})

print("\n--- Read-back (same thread) -------------------------------------")
print(followup["messages"][-1].content)


--- Read-back (same thread) -------------------------------------
The file contains 3 lines:

```
read
write
upload
```


FILESYSTEMBACKEND

In [86]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY3"]=os.getenv("GROQ_API_KEY3")
os.environ["OPENAI_API_KEY3"]=os.getenv("OPENAI_API_KEY3")
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")

In [87]:
from deepagents.backends import FilesystemBackend
ROOT = "."

agent=create_deep_agent(model="groq:qwen/qwen3.8-27b",backend=FilesystemBackend(root_dir=ROOT,virtual_mode=True))
print(f"✅ Agent created with FilesystemBackend(root_dir={ROOT!r}).")
print("   Files written by the agent will appear on your ACTUAL disk.")

✅ Agent created with FilesystemBackend(root_dir='.').
   Files written by the agent will appear on your ACTUAL disk.


In [89]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Create a file at /notes/todo.txt with exactly this content:\n"
            "1. Record video\n2. Edit video\n3. Upload video\n"
            "Then tell me you've done it."
        )
    }]
})

print("\n--- Agent reply -------------------------------------------------")
print(result["messages"][-1].content)


--- Agent reply -------------------------------------------------
Done — I've created `/notes/todo.txt` with the three items.


In [90]:
from pathlib import Path
print("\n--- Backend check (real filesystem) -----------------------------")
disk_path = Path(ROOT) / "notes" / "todo.txt"

if disk_path.exists():
    print(f"✅ FilesystemBackend is working — file exists on disk:")
    print(f"📄 {disk_path.resolve()}\n{'-' * 40}")
    print(disk_path.read_text())
else:
    print(f"⚠️  Expected file not found at {disk_path.resolve()}")
    print("    The agent may not have called the write tool, or the path "
          "mapping differs.")


--- Backend check (real filesystem) -----------------------------
✅ FilesystemBackend is working — file exists on disk:
📄 D:\agentic ai\DeepAgentsmodule\deepagentsdemo\notes\todo.txt
----------------------------------------
1. Record video
2. Edit video
3. Upload video


In [91]:
fresh_agent = create_deep_agent(
    model="groq:qwen/qwen3.8-27b",
    backend=FilesystemBackend(root_dir=ROOT, virtual_mode=True),
)

followup = fresh_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Read /notes/todo.txt ."
    }]
    # NOTE: no `files` state passed in — the file is read straight from disk
})

print("\n--- Read-back with a FRESH agent (proves disk persistence) ------")
print(followup["messages"][-1].content)



--- Read-back with a FRESH agent (proves disk persistence) ------
Here's the content of `/notes/todo.txt`:

1. Record video
2. Edit video
3. Upload video


Deep Agent — StoreBackend verification
Creates a deep agent backed by a LangGraph store, invokes it to write a file on one thread, then proves the backend works by reading that file back on a DIFFERENT thread — something StateBackend cannot do. """

In [ ]:
from langgraph.store.memory import InMemoryStore
from deepagents.backends import StoreBackend
store = InMemoryStore()

agent = create_deep_agent(
    model="groq:qwen/qwen3.8-27b",
    backend=StoreBackend(
        # Local dev: static namespace. No deployment runtime needed.
        # In a LangSmith Deployment you'd use the user-identity version.
        namespace=lambda rt: ("demo-user",),
    ),
    store=store,
)

print("✅ Agent created with StoreBackend + static namespace.")

✅ Agent created with StoreBackend + static namespace.


In [ ]:
import os
import uuid
# -----------------------------------------------------------------------------
# 2. THREAD 1 — write a file
# -----------------------------------------------------------------------------
thread_1 = {"configurable": {"thread_id": str(uuid.uuid4())}}

result = agent2.invoke({
    "messages": [{
        "role": "user",
        "content": """create a file at /notes/todo1.txt with exact same content:
1.read
2.write
3.upload
then tell u have done it"""
    }]
},
config=thread_1)

print("\n--- Agent reply (thread 1) --------------------------------------")
print(result["messages"][-1].content)


--- Agent reply (thread 1) --------------------------------------
Done! I've created the file at `/notes/todo1.txt` with the exact content:

```
1.read
2.write
3.upload
```


In [93]:
thread_2 = {"configurable": {"thread_id": str(uuid.uuid4())}}

followup = agent.invoke(
    {"messages": [{
        "role": "user",
        "content": "Read deepagentsdemo\notes\todo.txt "
    }]},
    config=thread_2,
)

print("\n--- Read-back on a different thread ---")
print(followup["messages"][-1].content)



--- Read-back on a different thread ---
Here's the content of `/notes/todo.txt`:

1. Record video
2. Edit video
3. Upload video
